# Xenium and Phenocycler Data Integration

Integrate Xenium spatial transcriptomics with Phenocycler imaging data:
- Load and preprocess Phenocycler data
- Spatial alignment
- Multi-modal integration
- Combined spatial analysis

**Input:** Xenium AnnData and Phenocycler data

**Output:** Integrated multi-modal dataset

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sc.settings.set_figure_params(dpi=80)
print("Libraries loaded successfully")

## 1. Load Xenium Data

In [ ]:
# Define paths
DATA_DIR = Path("../data/processed")
PHENOCYCLER_DIR = Path("../data/phenocycler")
OUTPUT_DIR = Path("../data/processed")
FIGURES_DIR = Path("../figures/06_integration")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PHENOCYCLER_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME = "xenium_sample_01"

# Load Xenium data
adata_xenium = sc.read_h5ad(DATA_DIR / f"{SAMPLE_NAME}_annotated.h5ad")

print(f"Xenium data loaded: {adata_xenium.shape}")
print(f"Cell types: {adata_xenium.obs['celltype'].nunique()}")

## 2. Load Phenocycler Data

Load Phenocycler/CODEX imaging data. Format depends on your specific data source.

In [ ]:
# Example: Load Phenocycler data
# Adjust based on your data format

# Option 1: If you have Phenocycler data as AnnData
try:
    adata_pheno = sc.read_h5ad(PHENOCYCLER_DIR / f"{SAMPLE_NAME}_phenocycler.h5ad")
    print(f"Phenocycler data loaded: {adata_pheno.shape}")
except FileNotFoundError:
    print("\nPhenocycler data file not found.")
    print("Please place Phenocycler data in ../data/phenocycler/")
    print("\nFor demonstration, creating placeholder...")
    
    # Create placeholder for demonstration
    adata_pheno = None

# Option 2: Load from CSV/TSV
# pheno_df = pd.read_csv(PHENOCYCLER_DIR / 'phenocycler_data.csv')
# adata_pheno = sc.AnnData(X=pheno_df.iloc[:, :n_markers].values)
# adata_pheno.obs = pheno_df[['cell_id', 'x', 'y', 'cluster']]
# adata_pheno.var_names = marker_names

## 3. Preprocess Phenocycler Data

In [ ]:
if adata_pheno is not None:
    # Basic QC
    sc.pp.filter_cells(adata_pheno, min_genes=3)
    sc.pp.filter_genes(adata_pheno, min_cells=5)
    
    # Normalize protein expression
    adata_pheno.layers['raw'] = adata_pheno.X.copy()
    sc.pp.normalize_total(adata_pheno, target_sum=1e4)
    sc.pp.log1p(adata_pheno)
    
    # PCA and UMAP
    sc.pp.scale(adata_pheno)
    sc.tl.pca(adata_pheno, n_comps=20)
    sc.pp.neighbors(adata_pheno, n_pcs=15)
    sc.tl.umap(adata_pheno)
    
    # Clustering
    sc.tl.leiden(adata_pheno, resolution=0.5, key_added='leiden')
    
    print(f"\nPhenocycler preprocessing complete")
    print(f"Clusters: {adata_pheno.obs['leiden'].nunique()}")
    
    # Visualize
    sc.pl.umap(adata_pheno, color='leiden')
    plt.savefig(FIGURES_DIR / 'phenocycler_umap.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Skipping Phenocycler preprocessing - no data loaded")

## 4. Spatial Alignment

Align Xenium and Phenocycler spatial coordinates

In [ ]:
if adata_pheno is not None and 'spatial' in adata_xenium.obsm and 'spatial' in adata_pheno.obsm:
    # Get spatial coordinates
    xenium_coords = adata_xenium.obsm['spatial']
    pheno_coords = adata_pheno.obsm['spatial']
    
    # Plot both modalities
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Xenium
    axes[0].scatter(xenium_coords[:, 0], xenium_coords[:, 1], 
                   s=1, alpha=0.5, c='blue')
    axes[0].set_title('Xenium Spatial Distribution')
    axes[0].set_xlabel('X')
    axes[0].set_ylabel('Y')
    
    # Phenocycler
    axes[1].scatter(pheno_coords[:, 0], pheno_coords[:, 1],
                   s=1, alpha=0.5, c='red')
    axes[1].set_title('Phenocycler Spatial Distribution')
    axes[1].set_xlabel('X')
    axes[1].set_ylabel('Y')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'spatial_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Spatial alignment visualization complete")
    print("\nNote: If coordinates don't align, apply transformation:")
    print("  - Translation")
    print("  - Rotation")
    print("  - Scaling")
else:
    print("Spatial coordinates not available for alignment")

## 5. Map Cells Between Modalities

In [ ]:
if adata_pheno is not None and 'spatial' in adata_xenium.obsm and 'spatial' in adata_pheno.obsm:
    from scipy.spatial import cKDTree
    
    # Find nearest Phenocycler cell for each Xenium cell
    tree = cKDTree(adata_pheno.obsm['spatial'])
    distances, indices = tree.query(adata_xenium.obsm['spatial'], k=1)
    
    # Add Phenocycler cluster info to Xenium
    adata_xenium.obs['nearest_pheno_cluster'] = adata_pheno.obs['leiden'].iloc[indices].values
    adata_xenium.obs['distance_to_pheno'] = distances
    
    # Filter for close matches (within threshold)
    distance_threshold = np.percentile(distances, 25)  # Keep closest 25%
    adata_xenium.obs['matched_pheno'] = distances < distance_threshold
    
    print(f"\nMatching statistics:")
    print(f"  Mean distance: {distances.mean():.2f}")
    print(f"  Median distance: {np.median(distances):.2f}")
    print(f"  Matched cells (<{distance_threshold:.2f}): {adata_xenium.obs['matched_pheno'].sum()}")
    
    # Visualize matching
    sc.pl.umap(adata_xenium, color=['celltype', 'nearest_pheno_cluster', 'distance_to_pheno'])
    plt.savefig(FIGURES_DIR / 'xenium_with_pheno_mapping.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Cell mapping requires spatial coordinates in both datasets")

## 6. Cross-Modal Analysis

In [ ]:
if adata_pheno is not None:
    # Compare cell type annotations
    matched_cells = adata_xenium.obs[adata_xenium.obs['matched_pheno']]
    
    if len(matched_cells) > 0:
        # Crosstab of Xenium cell types vs Phenocycler clusters
        cross_modal = pd.crosstab(
            matched_cells['celltype'],
            matched_cells['nearest_pheno_cluster'],
            normalize='index'
        ) * 100
        
        print("\nXenium cell types vs Phenocycler clusters (% overlap):")
        print(cross_modal)
        
        # Heatmap
        plt.figure(figsize=(10, 8))
        sns.heatmap(cross_modal, annot=True, fmt='.1f', cmap='YlOrRd')
        plt.title('Xenium Cell Types vs Phenocycler Clusters')
        plt.xlabel('Phenocycler Cluster')
        plt.ylabel('Xenium Cell Type')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'cross_modal_heatmap.png', dpi=300, bbox_inches='tight')
        plt.show()
    else:
        print("No matched cells found")
else:
    print("Cross-modal analysis requires Phenocycler data")

## 7. Integration with scVI (Advanced)

In [ ]:
# Multi-modal integration using scVI
# This requires matching features or use of totalVI

if adata_pheno is not None:
    print("For advanced multi-modal integration:")
    print("  - Use totalVI for joint RNA + protein modeling")
    print("  - Use scVI with shared features")
    print("  - Use MultiVI for multi-modal integration")
    print("\nExample:")
    print("  import scvi")
    print("  scvi.model.TOTALVI.setup_anndata(adata, protein_expression_obsm_key='protein_expression')")
    print("  vae = scvi.model.TOTALVI(adata)")
    print("  vae.train()")
else:
    print("Multi-modal integration requires Phenocycler data")

## 8. Save Integrated Data

In [ ]:
# Save Xenium data with Phenocycler annotations
output_file = OUTPUT_DIR / f"{SAMPLE_NAME}_integrated.h5ad"
adata_xenium.write_h5ad(output_file)

print(f"\nIntegrated Xenium data saved to: {output_file}")

if adata_pheno is not None:
    pheno_output = OUTPUT_DIR / f"{SAMPLE_NAME}_phenocycler_processed.h5ad"
    adata_pheno.write_h5ad(pheno_output)
    print(f"Phenocycler data saved to: {pheno_output}")

print("\nIntegration complete!")